In [0]:
# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("username", "valeriimatviiv", "2. Target Username")

catalog = dbutils.widgets.get("catalog")
username = dbutils.widgets.get("username")

bronze_schema = f"{catalog}.{username}_bronze"
silver_schema = f"{catalog}.{username}_silver"
gold_schema = f"{catalog}.{username}_gold"

# 1. Optimize and Z-Order Bronze Tables
print("--- Optimizing Bronze Tables ---")
spark.sql(f"OPTIMIZE {bronze_schema}.nasdaq_price_bronze ZORDER BY (Symbol)")
spark.sql(f"OPTIMIZE {bronze_schema}.finnhub_news_bronze ZORDER BY (id)")

# 2. Optimize and Z-Order Silver Tables
print("--- Optimizing Silver Tables ---")
spark.sql(f"OPTIMIZE {silver_schema}.nasdaq_price_silver ZORDER BY (Symbol, TradeDate)")
spark.sql(f"OPTIMIZE {silver_schema}.finnhub_news_silver ZORDER BY (Symbol, NewsDate)")

# 3. Optimize and Z-Order Gold Table
print("--- Optimizing Gold Table ---")
spark.sql(f"OPTIMIZE {gold_schema}.nasdaq_news_impact_gold ZORDER BY (Symbol, ImpactDate)")

# 4. Run Vacuum (Retention 168 Hours / 7 Days)
print("--- Vacuuming Stale Files ---")
spark.sql(f"VACUUM {silver_schema}.nasdaq_price_silver RETAIN 168 HOURS")
spark.sql(f"VACUUM {silver_schema}.finnhub_news_silver RETAIN 168 HOURS")
spark.sql(f"VACUUM {gold_schema}.nasdaq_news_impact_gold RETAIN 168 HOURS")

print("Maintenance routine completed successfully.")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# username = dbutils.widgets.get("username")
# gold_table = f"{catalog}.{username}_gold.nasdaq_news_impact_gold"

# print("--- History Log for Gold Table ---")
# display(spark.sql(f"DESCRIBE HISTORY {gold_table}"))